# Tahap 5 — Integrasi Dataset

**Tujuan notebook**
Menggabungkan data permukaan (Ogimet) dan data atmosfer terpilih (hasil Tahap 4) menjadi
satu dataset harian tunggal (`05_integrated_dataset.csv`), menggunakan
`03_daily_sounding_selection.csv` sebagai backbone kalender.

**Batasan cakupan (scope) — Tahap 5 TIDAK mencakup:**
- imputasi
- interpolasi
- filling missing values
- outlier removal
- labeling
- feature engineering
- scaling
- train/test split

**Deliverable:**
- `05_integrated_dataset.csv`
- `STAGE5_REPORT.md`


## 1. Import Library

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


## 2. Konfigurasi

In [2]:
INPUT_DIR = Path(".")
OGIMET_PATH = INPUT_DIR / "ogimet_standardized.csv"
SOUNDERPY_PATH = INPUT_DIR / "sounderpy_standardized.csv"
DAILY_SOUNDING_SELECTION_PATH = INPUT_DIR / "03_daily_sounding_selection.csv"

OUTPUT_CSV_PATH = Path("05_integrated_dataset.csv")
OUTPUT_REPORT_PATH = Path("STAGE5_REPORT.md")

OGIMET_COLS = ["date", "rr", "tavg", "rh"]
SOUNDERPY_ATMOS_COLS = ["cin", "kindex", "li", "tt", "sweat", "cape"]
SOUNDERPY_COLS = ["date", "hour"] + SOUNDERPY_ATMOS_COLS

OUTPUT_COLUMNS = [
    "date", "selected_hour", "selection_status",
    "rr", "tavg", "rh",
    "cin", "kindex", "li", "tt", "sweat", "cape",
]

HOUR_LABEL = {0: "00Z", 12: "12Z"}


## 3. Langkah 1: Load Data

`03_daily_sounding_selection.csv` dimuat sebagai backbone (sudah mencakup seluruh 2922
tanggal kalender). Ogimet dimuat dengan kolom fitur permukaan. SounderPy dimuat dengan
kolom `hour` (untuk validasi silang terhadap `selected_hour`, bukan sebagai join key) dan
enam variabel atmosfer.

**Catatan berkas mentah:** `ogimet_standardized.csv` mengandung baris kosong di ekor
berkas (seluruh kolom kosong, bukan bagian dari kalender 2017–2024 dan bukan record
tanggal apa pun). Baris semacam ini dibuang saat load karena tidak merepresentasikan
data — bukan operasi pembersihan/imputasi atas nilai yang hilang.

In [3]:
def load_backbone(path: Path) -> pd.DataFrame:
    """Load hasil Tahap 4 sebagai backbone integrasi (kalender lengkap)."""
    df = pd.read_csv(path)
    df["date"] = pd.to_datetime(df["date"], errors="raise")
    return df


def load_ogimet(path: Path) -> tuple[pd.DataFrame, int]:
    """Load data permukaan Ogimet, buang baris kosong tak bertanggal, konversi date."""
    df = pd.read_csv(path, usecols=OGIMET_COLS)
    n_raw = len(df)
    df = df[df["date"].notna()].copy()
    n_blank_rows_dropped = n_raw - len(df)
    df["date"] = pd.to_datetime(df["date"], errors="raise")
    return df, n_blank_rows_dropped


def load_sounderpy(path: Path) -> pd.DataFrame:
    """Load data atmosfer SounderPy dengan kolom yang relevan untuk integrasi."""
    df = pd.read_csv(path, usecols=SOUNDERPY_COLS)
    df["date"] = pd.to_datetime(df["date"], errors="raise")
    return df


backbone_df = load_backbone(DAILY_SOUNDING_SELECTION_PATH)
ogimet_df, n_ogimet_blank_rows_dropped = load_ogimet(OGIMET_PATH)
sounderpy_df = load_sounderpy(SOUNDERPY_PATH)

print(f"Backbone : {backbone_df.shape}")
print(f"Ogimet   : {ogimet_df.shape} (baris kosong tak bertanggal dibuang: {n_ogimet_blank_rows_dropped})")
print(f"SounderPy: {sounderpy_df.shape}")


Backbone : (2922, 3)
Ogimet   : (2922, 4) (baris kosong tak bertanggal dibuang: 365)
SounderPy: (2774, 8)


## 4. Langkah 2–3: Left Join Backbone ke Ogimet dan SounderPy

`date` digunakan sebagai primary key. Backbone (kalender lengkap, 2922 hari) tidak pernah
berkurang karena seluruh join adalah LEFT JOIN dari backbone.

In [4]:
def integrate(backbone: pd.DataFrame, ogimet: pd.DataFrame, sounderpy: pd.DataFrame) -> tuple[pd.DataFrame, int, int]:
    """Left join backbone terhadap Ogimet lalu SounderPy, berbasis date.

    Mengembalikan dataset gabungan beserta jumlah baris yang benar-benar menemukan
    pasangan (row match) pada masing-masing sumber, diukur lewat merge indicator —
    bukan lewat notna() pada nilai variabel, karena rr/tavg/rh/variabel atmosfer bisa
    saja NaN secara sah pada data mentah walau baris tanggalnya cocok (matched).
    """
    merged = backbone.merge(ogimet, on="date", how="left", validate="one_to_one", indicator="_ogimet_merge")
    n_ogimet_matched = int((merged["_ogimet_merge"] == "both").sum())
    merged = merged.drop(columns=["_ogimet_merge"])

    merged = merged.merge(sounderpy, on="date", how="left", validate="one_to_one", indicator="_sounderpy_merge")
    n_sounderpy_matched = int((merged["_sounderpy_merge"] == "both").sum())
    merged = merged.drop(columns=["_sounderpy_merge"])

    return merged, n_ogimet_matched, n_sounderpy_matched


assert len(ogimet_df) == len(backbone_df), (
    f"Jumlah baris Ogimet setelah pembersihan ({len(ogimet_df)}) tidak sama dengan "
    f"jumlah hari backbone ({len(backbone_df)})."
)

integrated_df, n_ogimet_matched, n_sounderpy_matched = integrate(backbone_df, ogimet_df, sounderpy_df)
integrated_df.head()


,date,selected_hour,selection_status,rr,tavg,rh,hour,cin,kindex,li,tt,sweat,cape
0,2017-01-01,12Z,SELECTED,NaN,28.2,83.9,12.0,-4.906,34.4,-4.510,42.2,219.162,2226.401
1,2017-01-02,12Z,SELECTED,26.9,26.5,87.4,12.0,-22.887,35.2,-2.955,41.5,223.381,921.164
2,2017-01-03,12Z,SELECTED,78.0,26.2,87.6,12.0,0.000,36.4,-3.317,40.5,277.368,1671.572
3,2017-01-04,12Z,SELECTED,33.0,24.7,92.4,12.0,-4.865,35.1,-0.752,39.0,294.560,397.409
4,2017-01-05,12Z,SELECTED,83.0,25.7,88.3,12.0,-15.811,38.0,-5.124,44.1,299.131,2064.721


## 5. Validasi Silang: `hour` (SounderPy) vs `selected_hour` (Tahap 4)

Karena SounderPy saat ini memiliki maksimal satu sounding per tanggal (dikonfirmasi pada
Tahap 3), left join berbasis `date` sudah cukup untuk membawa jam yang benar tanpa perlu
join gabungan `date + hour`. Validasi berikut memastikan asumsi itu tetap berlaku pada
data aktual, sebelum kolom `hour` dibuang (tidak termasuk kolom output).

In [5]:
def validate_hour_consistency(df: pd.DataFrame) -> dict:
    """Pastikan hour hasil join konsisten dengan selected_hour dan selection_status."""
    df = df.copy()
    df["hour_label"] = df["hour"].map(HOUR_LABEL)

    selected_rows = df[df["selection_status"] == "SELECTED"]
    no_sounding_rows = df[df["selection_status"] == "NO_SOUNDING"]

    mismatch_selected = selected_rows[selected_rows["hour_label"] != selected_rows["selected_hour"]]
    mismatch_no_sounding = no_sounding_rows[no_sounding_rows["hour_label"].notna()]

    return {
        "n_mismatch_selected": len(mismatch_selected),
        "n_mismatch_no_sounding": len(mismatch_no_sounding),
        "mismatch_selected_examples": mismatch_selected["date"].head(5).tolist(),
        "mismatch_no_sounding_examples": mismatch_no_sounding["date"].head(5).tolist(),
    }


hour_validation = validate_hour_consistency(integrated_df)
hour_validation


{'n_mismatch_selected': 0,
 'n_mismatch_no_sounding': 0,
 'mismatch_selected_examples': [],
 'mismatch_no_sounding_examples': []}

In [6]:
hour_consistency_ok = (
    hour_validation["n_mismatch_selected"] == 0 and hour_validation["n_mismatch_no_sounding"] == 0
)
assert hour_consistency_ok, "Ditemukan ketidakcocokan hour vs selected_hour — perlu investigasi manual."
print("Validasi hour vs selected_hour LULUS.")

integrated_df = integrated_df.drop(columns=["hour"])


Validasi hour vs selected_hour LULUS.


## 6. Susun Kolom Output

In [7]:
integrated_df = integrated_df[OUTPUT_COLUMNS]
integrated_df.head()


,date,selected_hour,selection_status,rr,tavg,rh,cin,kindex,li,tt,sweat,cape
0,2017-01-01,12Z,SELECTED,NaN,28.2,83.9,-4.906,34.4,-4.510,42.2,219.162,2226.401
1,2017-01-02,12Z,SELECTED,26.9,26.5,87.4,-22.887,35.2,-2.955,41.5,223.381,921.164
2,2017-01-03,12Z,SELECTED,78.0,26.2,87.6,0.000,36.4,-3.317,40.5,277.368,1671.572
3,2017-01-04,12Z,SELECTED,33.0,24.7,92.4,-4.865,35.1,-0.752,39.0,294.560,397.409
4,2017-01-05,12Z,SELECTED,83.0,25.7,88.3,-15.811,38.0,-5.124,44.1,299.131,2064.721


## 7. Validasi Struktural

In [8]:
def validate_structure(df: pd.DataFrame, n_ogimet_matched: int, n_sounderpy_matched: int, expected_days: int) -> dict:
    """Validasi integritas struktural dataset terintegrasi."""
    n_rows = len(df)
    n_cols = len(df.columns)
    n_duplicate_dates = int(df["date"].duplicated().sum())
    n_unique_dates = df["date"].nunique()

    return {
        "n_rows": n_rows,
        "n_cols": n_cols,
        "n_duplicate_dates": n_duplicate_dates,
        "n_unique_dates": n_unique_dates,
        "n_ogimet_row_matched": n_ogimet_matched,
        "n_sounderpy_available": n_sounderpy_matched,
        "expected_days": expected_days,
    }


structural_result = validate_structure(
    integrated_df, n_ogimet_matched, n_sounderpy_matched, expected_days=len(backbone_df)
)
structural_result


{'n_rows': 2922,
 'n_cols': 12,
 'n_duplicate_dates': 0,
 'n_unique_dates': 2922,
 'n_ogimet_row_matched': 2922,
 'n_sounderpy_available': 2774,
 'expected_days': 2922}

## 8. Audit Wajib

Total row, total column, duplicate date, jumlah SELECTED / NO_SOUNDING, dan jumlah
missing pada setiap variabel SounderPy.

In [9]:
def compute_audit(df: pd.DataFrame) -> dict:
    """Hitung ringkasan audit wajib Tahap 5."""
    sounderpy_vars = ["cin", "kindex", "li", "tt", "sweat", "cape"]
    missing_per_var = {col: int(df[col].isna().sum()) for col in sounderpy_vars}

    return {
        "total_row": len(df),
        "total_column": len(df.columns),
        "duplicate_date": int(df["date"].duplicated().sum()),
        "n_selected": int((df["selection_status"] == "SELECTED").sum()),
        "n_no_sounding": int((df["selection_status"] == "NO_SOUNDING").sum()),
        "missing_per_var": missing_per_var,
    }


audit_result = compute_audit(integrated_df)

print(f"Total row      : {audit_result['total_row']}")
print(f"Total column   : {audit_result['total_column']}")
print(f"Duplicate date : {audit_result['duplicate_date']}")
print(f"Jumlah SELECTED   : {audit_result['n_selected']}")
print(f"Jumlah NO_SOUNDING: {audit_result['n_no_sounding']}")
print("Missing per variabel SounderPy:")
for var, jumlah in audit_result["missing_per_var"].items():
    print(f"  {var}: {jumlah}")


Total row      : 2922
Total column   : 12
Duplicate date : 0
Jumlah SELECTED   : 2774
Jumlah NO_SOUNDING: 148
Missing per variabel SounderPy:
  cin: 149
  kindex: 194
  li: 194
  tt: 194
  sweat: 194
  cape: 149


## 9. Simpan Output CSV

In [10]:
integrated_df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"Tersimpan: {OUTPUT_CSV_PATH.resolve()}")


Tersimpan: /home/claude/work/05_integrated_dataset.csv


## 10. Cek Acceptance Criteria

In [11]:
def check_acceptance_criteria(audit: dict, structural: dict) -> pd.DataFrame:
    """Bandingkan hasil aktual terhadap acceptance criteria Tahap 5."""
    rows = [
        ("total row = 2922", audit["total_row"] == 2922, audit["total_row"]),
        ("duplicate date = 0", audit["duplicate_date"] == 0, audit["duplicate_date"]),
        ("SELECTED = 2774", audit["n_selected"] == 2774, audit["n_selected"]),
        ("NO_SOUNDING = 148", audit["n_no_sounding"] == 148, audit["n_no_sounding"]),
        (
            "seluruh tanggal kalender tetap ada",
            structural["n_unique_dates"] == structural["expected_days"],
            structural["n_unique_dates"],
        ),
        (
            "tidak ada tanggal yang hilang",
            structural["n_rows"] == structural["expected_days"],
            structural["n_rows"],
        ),
        (
            "variabel Ogimet tersedia untuk 2922 hari",
            structural["n_ogimet_row_matched"] == 2922,
            structural["n_ogimet_row_matched"],
        ),
        (
            "variabel SounderPy tersedia untuk 2774 hari",
            structural["n_sounderpy_available"] == 2774,
            structural["n_sounderpy_available"],
        ),
    ]
    return pd.DataFrame(rows, columns=["kriteria", "terpenuhi", "nilai_aktual"])


acceptance_df = check_acceptance_criteria(audit_result, structural_result)
acceptance_df


,kriteria,terpenuhi,nilai_aktual
0,total row = 2922,True,2922
1,duplicate date = 0,True,0
2,SELECTED = 2774,True,2774
3,NO_SOUNDING = 148,True,148
4,seluruh tanggal kalender tetap ada,True,2922
5,tidak ada tanggal yang hilang,True,2922
6,variabel Ogimet tersedia untuk 2922 hari,True,2922
7,variabel SounderPy tersedia untuk 2774 hari,True,2774


## 11. Susun STAGE5_REPORT.md

In [12]:
def build_stage5_report(audit: dict, structural: dict, hour_validation: dict, acceptance_df: pd.DataFrame, n_ogimet_blank_rows_dropped: int) -> str:
    """Bangun konten STAGE5_REPORT.md dari hasil audit dan validasi aktual."""
    missing_lines = "\n".join(
        f"- {var}: {jumlah}" for var, jumlah in audit["missing_per_var"].items()
    )

    acceptance_lines = "\n".join(
        f"- [{'x' if row.terpenuhi else ' '}] {row.kriteria} (aktual: {row.nilai_aktual})"
        for row in acceptance_df.itertuples(index=False)
    )

    hour_status = (
        "LULUS"
        if hour_validation["n_mismatch_selected"] == 0 and hour_validation["n_mismatch_no_sounding"] == 0
        else "GAGAL"
    )

    report = f"""# STAGE5_REPORT — Integrasi Dataset

## Ringkasan

- Total row     : {audit['total_row']}
- Total column  : {audit['total_column']}
- Duplicate date: {audit['duplicate_date']}
- Jumlah SELECTED   : {audit['n_selected']}
- Jumlah NO_SOUNDING : {audit['n_no_sounding']}

## Missing per Variabel SounderPy

{missing_lines}

## Hasil Validasi Konsistensi

- Validasi silang hour (SounderPy) vs selected_hour (Tahap 4): {hour_status}
  - Mismatch pada baris SELECTED   : {hour_validation['n_mismatch_selected']}
  - Mismatch pada baris NO_SOUNDING: {hour_validation['n_mismatch_no_sounding']}
- Validasi struktural (jumlah baris, duplicate date, keutuhan tanggal kalender): {'LULUS' if structural['n_rows'] == structural['expected_days'] and structural['n_duplicate_dates'] == 0 else 'GAGAL'}

## Acceptance Criteria

{acceptance_lines}

## Catatan Berkas Mentah

`ogimet_standardized.csv` mengandung {n_ogimet_blank_rows_dropped} baris kosong tak
bertanggal di ekor berkas (bukan record kalender). Baris tersebut dibuang sebelum join
karena bukan data, bukan hasil pembersihan/imputasi atas nilai yang hilang.

## Catatan Cakupan

Tahap ini murni LEFT JOIN berbasis date terhadap backbone hasil Tahap 4. Tidak ada
imputasi, interpolasi, filling missing values, outlier removal, labeling, feature
engineering, scaling, maupun train/test split yang dilakukan. Nilai kosong pada variabel
SounderPy untuk hari tanpa sounding (`NO_SOUNDING`) dibiarkan sebagai NaN apa adanya.
Dataset ini adalah input resmi untuk Tahap 6 (Quality Assessment).
"""
    return report


report_content = build_stage5_report(
    audit_result, structural_result, hour_validation, acceptance_df, n_ogimet_blank_rows_dropped
)
OUTPUT_REPORT_PATH.write_text(report_content)
print(f"Tersimpan: {OUTPUT_REPORT_PATH.resolve()}")
print()
print(report_content)


Tersimpan: /home/claude/work/STAGE5_REPORT.md

# STAGE5_REPORT — Integrasi Dataset

## Ringkasan

- Total row     : 2922
- Total column  : 12
- Duplicate date: 0
- Jumlah SELECTED   : 2774
- Jumlah NO_SOUNDING : 148

## Missing per Variabel SounderPy

- cin: 149
- kindex: 194
- li: 194
- tt: 194
- sweat: 194
- cape: 149

## Hasil Validasi Konsistensi

- Validasi silang hour (SounderPy) vs selected_hour (Tahap 4): LULUS
  - Mismatch pada baris SELECTED   : 0
  - Mismatch pada baris NO_SOUNDING: 0
- Validasi struktural (jumlah baris, duplicate date, keutuhan tanggal kalender): LULUS

## Acceptance Criteria

- [x] total row = 2922 (aktual: 2922)
- [x] duplicate date = 0 (aktual: 0)
- [x] SELECTED = 2774 (aktual: 2774)
- [x] NO_SOUNDING = 148 (aktual: 148)
- [x] seluruh tanggal kalender tetap ada (aktual: 2922)
- [x] tidak ada tanggal yang hilang (aktual: 2922)
- [x] variabel Ogimet tersedia untuk 2922 hari (aktual: 2922)
- [x] variabel SounderPy tersedia untuk 2774 hari (aktual: 2774)

##